# Notebook 03: Pan/Tilt Camera Control

## ADAS Connection
A self-driving car needs to see in multiple directions. Modern vehicles use **adaptive cameras and sensors** that can physically move to track objects -- like a camera that follows a pedestrian stepping off a curb, or headlights that turn with the steering wheel to illuminate a curve.

Your robot has a **2DOF (two degrees of freedom) pan/tilt camera mount** -- it can look left/right (pan) and up/down (tilt). This is the same concept used in **360-degree camera systems** and **automated security cameras** in real ADAS applications.

---

## How It Works
Two servos control the camera position:
- **Horizontal servo (pan)** -- rotates the camera left and right
- **Vertical servo (tilt)** -- rotates the camera up and down

Each servo takes an **angle value from 0 to 180 degrees**. The center position is 90 degrees -- straight ahead.

The servos use **PWM (Pulse Width Modulation)** to hold their position. The duty cycle of the PWM signal tells the servo what angle to move to.

| Angle | Position |
|-------|----------|
| 0     | Full left / Full down |
| 90    | Center |
| 180   | Full right / Full up |

---

## The Code
Run this cell first to set up the servo pins.

In [1]:
import RPi.GPIO as GPIO
import time

# Pin definitions (BCM numbering)
HORIZONTAL = 9    # swapped
VERTICAL   = 11   # swapped

GPIO.setmode(GPIO.BCM)
GPIO.setwarnings(False)
GPIO.setup(HORIZONTAL, GPIO.OUT)
GPIO.setup(VERTICAL,   GPIO.OUT)

pwm_horizontal = GPIO.PWM(HORIZONTAL, 50)
pwm_vertical   = GPIO.PWM(VERTICAL,   50)
pwm_horizontal.start(0)
pwm_vertical.start(0)
time.sleep(0.5)   # let PWM settle before moving


def angle_to_duty(angle):
    return 2.5 + (angle / 180.0) * 10.0

def move_to(pwm, angle, delay=0.8):
    angle = max(0, min(180, angle))
    pwm.ChangeDutyCycle(angle_to_duty(angle))
    time.sleep(delay)
    pwm.ChangeDutyCycle(0)

def pan(angle, delay=0.8):
    """Pan is physically inverted -- flip the angle"""
    move_to(pwm_horizontal, 180 - angle, delay)

def tilt(angle, delay=0.8):
    move_to(pwm_vertical, angle, delay)

def center():
    pan(90)
    tilt(90)

print('Pan/tilt ready!')
print('Centering camera...')
center()

Pan/tilt ready!
Centering camera...


---

## YOUR TURN -- Tweak Zone 1: Move the Camera

Change the angle values and run the cell. Watch where the camera points.

- **0** = full left / full down
- **90** = center
- **180** = full right / full up

> **Think like an engineer:** Why is it important that the camera returns to center after a scan? What would happen in a real car if the camera stayed pointed to the side?

In [2]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
PAN_ANGLE  = 90   #left/right: 0=left, 90=center, 180=right
TILT_ANGLE = 120    #up/down:    0=up, 90=center, 180=down
# ═══════════════════════════════════════
print(f'Moving camera -- Pan:{PAN_ANGLE} Tilt:{TILT_ANGLE}')
pan(PAN_ANGLE)
tilt(TILT_ANGLE)
print('Done.')

Moving camera -- Pan:90 Tilt:120
Done.


---

## YOUR TURN -- Tweak Zone 2: Scan Speed

The camera can scan left and right automatically. Change the speed of the scan.

- `SCAN_DELAY` controls how long the servo waits at each position before moving
- A smaller value = faster scan, larger value = slower scan

> **Think like an engineer:** How fast should a self-driving car scan its surroundings? What are the tradeoffs between scanning fast vs slow?

In [3]:
# ═══════════════════════════════════════
#   TWEAK THESE VALUES
SCAN_DELAY = 0.8    # seconds at each position -- try 0.3, 0.5, 0.8, 1.5
LEFT_LIMIT  = 30    # how far left to scan (0-90)
RIGHT_LIMIT = 150   # how far right to scan (90-180)
# ═══════════════════════════════════════

center()
print('Scanning left...')
pan(LEFT_LIMIT,  SCAN_DELAY)
print('Scanning right...')
pan(RIGHT_LIMIT, SCAN_DELAY)
print('Returning to center...')
pan(90, SCAN_DELAY)
print('Scan complete.')

Scanning left...
Scanning right...
Returning to center...
Scan complete.


---

## YOUR TURN -- Tweak Zone 3: Search Pattern

Real ADAS cameras do not just scan left and right -- they follow a search pattern to cover the whole scene. Program a search pattern that scans both horizontally and vertically.

We have started the pattern for you -- add more positions to cover more of the scene.

> **Think like an engineer:** This is called a **coverage scan**. Parking assist cameras use this to build a complete picture of the space around the car. What pattern would give the best coverage?

In [4]:
# ═══════════════════════════════════════
#   TWEAK THE POSITIONS AND ADD MORE
MOVE_DELAY = 0.6    # seconds at each position

# Each entry is (pan_angle, tilt_angle)
SCAN_PATTERN = [
    (90,  90),   # center
    (30,  90),   # left
    (30,  120),  # left up
    (90,  120),  # center up
    (150, 120),  # right up
    (150, 90),   # right
    (90,  90),   # back to center
    # ADD MORE POSITIONS HERE
]
# ═══════════════════════════════════════

print('Running search pattern...')
for i, (ipan, itilt) in enumerate(SCAN_PATTERN):
    print(f'  Position {i+1}: Pan={ipan} Tilt={itilt}')
    pan(ipan,  MOVE_DELAY)
    tilt(itilt, MOVE_DELAY)

print('Pattern complete.')

Running search pattern...
  Position 1: Pan=90 Tilt=90
  Position 2: Pan=30 Tilt=90
  Position 3: Pan=30 Tilt=120
  Position 4: Pan=90 Tilt=120
  Position 5: Pan=150 Tilt=120
  Position 6: Pan=150 Tilt=90
  Position 7: Pan=90 Tilt=90
Pattern complete.


---

## What Happened?

Think about these questions with your team:

1. Did the camera reach all the positions you expected? Were any limited by the physical mount?
2. What scan pattern gave the best coverage of the room?
3. How would you combine this with the ultrasonic sensor to build a map of obstacles around the robot?

---

## CHALLENGE -- Advanced Students

Write a function called `smooth_scan()` that moves the camera gradually from one position to another in small steps rather than jumping directly. This is called **interpolation** and makes the movement look much more natural.

Hint: use a loop with small angle increments and a short delay between each step.

In [11]:
# YOUR CODE HERE
# ═══════════════════════════════════════
STEP_SIZE  = 5      # degrees per step
STEP_DELAY = 0.05   # seconds between steps
# ═══════════════════════════════════════

print('Running search pattern...')
#for i, (ipan, itilt) in enumerate(SCAN_PATTERN):
#    print(f'  Position {i+1}: Pan={ipan} Tilt={itilt}')
#    pan(ipan,  MOVE_DELAY)
#    tilt(itilt, MOVE_DELAY)

center()
def smooth_scan(start_angle, end_angle, step_size=STEP_SIZE, step_delay=STEP_DELAY):
    for i in range (start_angle, end_angle, STEP_SIZE):
        pan(i, STEP_DELAY)
        print(f'  Position {i}: Pan={i}')
    pass

smooth_scan(30,151,2,0.03)
print('Pattern complete.')
center()



Running search pattern...
  Position 30: Pan=30
  Position 35: Pan=35
  Position 40: Pan=40
  Position 45: Pan=45
  Position 50: Pan=50
  Position 55: Pan=55
  Position 60: Pan=60
  Position 65: Pan=65
  Position 70: Pan=70
  Position 75: Pan=75
  Position 80: Pan=80
  Position 85: Pan=85
  Position 90: Pan=90
  Position 95: Pan=95
  Position 100: Pan=100
  Position 105: Pan=105
  Position 110: Pan=110
  Position 115: Pan=115
  Position 120: Pan=120
  Position 125: Pan=125
  Position 130: Pan=130
  Position 135: Pan=135
  Position 140: Pan=140
  Position 145: Pan=145
  Position 150: Pan=150
Pattern complete.


---

## Always clean up when you are done!

In [12]:
center()
pwm_horizontal.stop()
pwm_vertical.stop()
GPIO.cleanup()
print('GPIO cleaned up.')

GPIO cleaned up.
